In [ ]:
!pip install earthpy gdal --quiet
!pip install torch torchvision segmentation-models-pytorch --quiet

In [ ]:
from osgeo import gdal, gdal_array
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import io
import joblib
import torch
import cv2
import segmentation_models_pytorch as smp
import pandas as pd

pca = joblib.load('/kaggle/input/pca/other/default/1/pca_model.joblib')
scaler = joblib.load('/kaggle/input/pca/other/default/1/scaler_model.joblib')


In [ ]:
def apply_pca(file_name,is_file_name):
    # === Select and Load a New Image ===
    if(is_file_name):
        new_image_path = os.path.join(data_path, file_name)  # pick one test image
    else:
        new_image_path=file_name
        
    img_ds = gdal.Open(new_image_path, gdal.GA_ReadOnly)
    
    # === Read Band 3 and 4 ===
    band3 = img_ds.GetRasterBand(3).ReadAsArray()
    band4 = img_ds.GetRasterBand(4).ReadAsArray()
    
    # === Stack and Reshape ===
    img_array = np.stack([band3, band4], axis=-1)  # shape: (H, W, 2)
    h, w, _ = img_array.shape
    reshaped = img_array.reshape(-1, 2)
    
    # === Transform with Scaler and PCA ===
    scaled = scaler.transform(reshaped)
    pca_output = pca.transform(scaled)
    
    # === Select First Principal Component and Reshape to Image ===
    pc1 = pca_output[:, 0].reshape(h, w)
    return pc1


In [ ]:
def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    Returns:
        str: RLE-encoded string, or a single space " " if mask is all zeros.
    """
    if np.sum(mask) == 0:
        return " "  # As it seems that kaggle reject nulls. We'll handle cloud-free images with empty spaces.
    
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed

    return " ".join(map(str, runs))  # Convert to string format


In [ ]:
def rle_decode(mask_rle: str, shape=(256, 256)) -> np.ndarray:
    """Decodes an RLE-encoded string into a binary mask with validation checks."""
    
    if not isinstance(mask_rle, str) or not mask_rle.strip() or mask_rle.lower() == 'nan':
        # Return all-zero mask if RLE is empty, invalid, or NaN
        return np.zeros(shape, dtype=np.uint8)
    
    try:
        s = list(map(int, mask_rle.split()))
    except:
        raise Exception("RLE segmentation must be a string and containing only integers")
    
    if len(s) % 2 != 0:
        raise Exception("RLE segmentation must have even-length (start, length) pairs")
    
    if any(x < 0 for x in s):
        raise Exception("RLE segmentation must not contain negative values")
    
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    starts, lengths = s[0::2], s[1::2]
    
    for start, length in zip(starts, lengths):
        if start >= mask.size or start + length > mask.size:
            raise Exception("RLE indices exceed image size")
        mask[start:start + length] = 1
    
    return mask.reshape(shape, order='F')  # Convert to column-major order


In [ ]:
def generate_csv(model,files):
    df = pd.read_csv('/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/sample_submission.csv', dtype={'id': str})
    for file in files:
        path=os.path.join(test_dir,file)
        img_pca=apply_pca(path,False)
        img_pca = cv2.resize(img_pca, (256, 256), interpolation=cv2.INTER_LINEAR)
        eps = 1e-7
        img = (img_pca.astype(np.float32) - img_pca.min()) / (img_pca.max() - img_pca.min() + eps)
        # Expand channel dimension: [H, W] → [1, H, W]
        img = np.expand_dims(img, axis=0)
        img = np.expand_dims(img, axis=0)
        img=torch.from_numpy(img)
        # prediction
        mask=model(img)
        mask=torch.sigmoid(mask)  # (B, 1, H, W)
        mask = (mask > 0.5).cpu().numpy().astype(np.uint8)  
        img=img[0,0].cpu().numpy()
        mask=mask[0,0]
        # mask encoding to string
        mask_string=rle_encode(mask)
        decoded_mask=rle_decode(mask_string)
        assert np.all(decoded_mask == mask)
        file_id = file.replace(".tif", "")
        df.loc[df['id'] == file_id, 'segmentation'] = mask_string
        # display the image and predicted mask
        fig, axs = plt.subplots(1, 2, figsize=(10, 5))
        axs[0].imshow(img, cmap='gray')
        axs[0].set_title('Image')
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)        
        axs[1].set_title('Predicted Mask')
        plt.show()
    df.to_csv('submission.csv', index=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = smp.Unet(in_channels=1, out_channels=1)
model = torch.load('/kaggle/input/eff-unet-0.86/other/default/1/eff_unet_model.pth')
model.eval() 

In [ ]:
test_dir="/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data"
test_files=os.listdir(test_dir)
print(len(test_files))

In [ ]:
generate_csv(model,test_files)